## kagglehub dataset import
https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017/data

In [20]:
import kagglehub
import os
import pandas as pd
import shutil

# import data with kagglehub
path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")
print(path)
print(os.listdir(path))

# copy the freshly downloaded files into your project's data/raw folder
raw_dir = "data/raw"
os.makedirs(raw_dir, exist_ok=True)
for fname in ["results.csv", "shootouts.csv"]:
    shutil.copy(os.path.join(path, fname), os.path.join(raw_dir, fname))

C:\Users\ARW\.cache\kagglehub\datasets\martj42\international-football-results-from-1872-to-2017\versions\132
['former_names.csv', 'goalscorers.csv', 'results.csv', 'shootouts.csv']


In [ ]:
# import the results.csv data into a pandas dataframe and filter for matches after 1993-01-01
df_results = pd.read_csv(os.path.join(path, "results.csv"))
df_results = df_results[df_results["date"] >= "1993-01-01"]
df_results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
18708,1993-01-01,Ghana,Mali,1.0,1.0,Friendly,Libreville,Gabon,True
18709,1993-01-02,Gabon,Burkina Faso,1.0,1.0,Friendly,Libreville,Gabon,False
18710,1993-01-02,Kuwait,Lebanon,2.0,0.0,Friendly,Kuwait City,Kuwait,False
18711,1993-01-03,Burkina Faso,Mali,1.0,0.0,Friendly,Libreville,Gabon,True
18712,1993-01-03,Gabon,Ghana,2.0,3.0,Friendly,Libreville,Gabon,False


In [ ]:
# import the shootouts.csv data into a pandas dataframe
df_shootouts = pd.read_csv(os.path.join(path, "shootouts.csv"))
df_shootouts.head()

,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN


In [ ]:
# number of tournament types in the dataset
df_results["tournament"].value_counts().head(50)

tournament
Friendly                                 10211
FIFA World Cup qualification              6821
UEFA Euro qualification                   1991
African Cup of Nations qualification      1745
UEFA Nations League                        658
AFC Asian Cup qualification                613
African Cup of Nations                     606
FIFA World Cup                             597
CFU Caribbean Cup qualification            453
CONCACAF Nations League                    422
Gold Cup                                   404
CECAFA Cup                                 383
Island Games                               368
COSAFA Cup                                 354
Copa América                               352
UEFA Euro                                  308
AFF Championship                           291
AFC Asian Cup                              282
Gulf Cup                                   234
CFU Caribbean Cup                          187
SAFF Cup                                   162
UN

In [ ]:
# list of unique tournament types in the dataset
sorted(df_results["tournament"].unique())

['ABCS Tournament',
 'AFC Asian Cup',
 'AFC Asian Cup qualification',
 'AFC Challenge Cup',
 'AFC Challenge Cup qualification',
 'AFC Solidarity Cup',
 'AFF Championship',
 'AFF Championship qualification',
 'ASEAN Championship',
 'ASEAN Championship qualification',
 'African Cup of Nations',
 'African Cup of Nations qualification',
 'Afro-Asian Games',
 'Al Ain International Cup',
 'Amílcar Cabral Cup',
 'Arab Cup',
 'Arab Cup qualification',
 'Asian Games',
 'Atlantic Heritage Cup',
 'Baltic Cup',
 'Benedikt Fontana Cup',
 'CAFA Nations Cup',
 'CECAFA Cup',
 'CFU Caribbean Cup',
 'CFU Caribbean Cup qualification',
 'CONCACAF Nations League',
 'CONCACAF Nations League qualification',
 'CONCACAF Series',
 'CONIFA Africa Football Cup',
 'CONIFA Asia Cup',
 'CONIFA European Football Cup',
 'CONIFA South America Football Cup',
 'CONIFA World Cup qualification',
 'CONIFA World Football Cup',
 'CONIFA World Football Cup qualification',
 'CONMEBOL–UEFA Cup of Champions',
 'COSAFA Cup',
 'COS

In [ ]:
# number of unique tournament types in the dataset
df_results["tournament"].nunique()

142

In [40]:
# merge the shootouts data into the results data
df_results = pd.read_csv("data/raw/results.csv")
df_shootouts = pd.read_csv("data/raw/shootouts.csv")

df_results = df_results[df_results["date"] >= "1993-01-01"]

df_FINAL_RESULTS = df_results.merge(
    df_shootouts[
        ["date", "home_team", "away_team", "winner"]
    ],
    on=["date", "home_team", "away_team"],
    how="left",
)

In [41]:
df_FINAL_RESULTS["winner"].notna().sum()

np.int64(511)

In [43]:
# No duplicate matches created by merge
len(df_FINAL_RESULTS) == len(df_results)

# Number of shootout matches
df_FINAL_RESULTS["winner"].notna().sum()

# Inspect a few shootout matches
df_FINAL_RESULTS.loc[
    df_FINAL_RESULTS["winner"].notna(),
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "winner",
    ],
].head(20)

,date,home_team,away_team,home_score,away_score,winner
29,1993-01-23,Japan,Switzerland,1.0,1.0,Switzerland
80,1993-02-20,Finland,Estonia,0.0,0.0,Finland
86,1993-02-24,Argentina,Denmark,1.0,1.0,Argentina
360,1993-05-27,Bolivia,Paraguay,2.0,1.0,Bolivia
362,1993-05-28,Martinique,Saint Kitts and Nevis,1.0,1.0,Martinique
370,1993-05-30,Jamaica,Martinique,0.0,0.0,Martinique
447,1993-06-17,Singapore,Myanmar,3.0,3.0,Singapore
479,1993-06-26,Colombia,Uruguay,1.0,1.0,Colombia
481,1993-06-27,Brazil,Argentina,1.0,1.0,Argentina
488,1993-07-01,Colombia,Argentina,0.0,0.0,Argentina


In [44]:
df_FINAL_RESULTS.to_parquet(
    "../data/processed/matches.parquet",
    index=False,
)